In [2]:
import gradio as gr
def greet(name):
  return "Hello " + name + "!"

demo = gr.Interface(fn=greet, inputs="text", outputs="text")
demo.launch()

c:\Users\ABHCST9\Desktop\dev\workspace\next_ai\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [3]:
import torch
import torchvision.models as models
from torchvision.models import ResNet50_Weights

# 사전학습 모델 로드 (torch.hub 안 씀)
weights = ResNet50_Weights.IMAGENET1K_V2
# eval: 추가 학습을 막기
model = models.resnet50(weights=weights).eval()

# 라벨도 weights에서 바로 가져옴 (requests로 다운받을 필요 없음)
labels = weights.meta["categories"]

# 전처리도 weights에 내장되어 있음
preprocess = weights.transforms()

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\ABHCST9/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:08<00:00, 11.7MB/s]


In [4]:
# 주어진 이미지에 대한 예측
def predict(inp):
  # unsqueeze 0번째 축에 []로 감싸기(model이 원하는 형태이므로)
  inp = preprocess(inp).unsqueeze(0)

  with torch.no_grad():
    prediction = torch.nn.functional.softmax(model(inp)[0], dim=0)
    confidences = {labels[i]: float(prediction[i]) for i in range(len(labels))}
  return confidences

In [5]:
import requests
import os

# 이미지를 다운로드
def download_image(url, save_path):
  headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
  }
  try:
    response = requests.get(url, stream=True, headers=headers, verify=False)
    response.raise_for_status()
    with open(save_path, 'wb') as file:
      for chunk in response.iter_content(chunk_size=8192):
        file.write(chunk)
  except requests.exceptions.RequestException as e:
    print(f"오류 발생: {e}")
# 예제 이미지 URL
file_path = {
    "lion.jpg": "https://upload.wikimedia.org/wikipedia/commons/7/73/Lion_waiting_in_Namibia.jpg",
    "plane.jpg": "https://upload.wikimedia.org/wikipedia/commons/e/e5/Airbus_A350-941_F-WZGG_MSN002_ZWS_2018-02-14.jpg"
}
# 예제 이미지 다운로드
for i in file_path:
    download_image(file_path[i], i)

c:\Users\ABHCST9\Desktop\dev\workspace\next_ai\venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'upload.wikimedia.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\ABHCST9\Desktop\dev\workspace\next_ai\venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'upload.wikimedia.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


오류 발생: 404 Client Error: Not Found for url: https://upload.wikimedia.org/wikipedia/commons/e/e5/Airbus_A350-941_F-WZGG_MSN002_ZWS_2018-02-14.jpg


In [6]:
import gradio as gr

# Gradio interface 실행
gr.Interface(
  fn=predict,
  inputs=gr.Image(type="pil"),
  outputs=gr.Label(num_top_classes=3),
  examples=["lion.jpg", "plane.jpg"]
).launch(share=True)


* Running on local URL:  http://127.0.0.1:7861

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt

# 데이터 전처리를 위한 변환
# Compose는 transforms의 처리를 하나로 묶어줌
# 기존의 형태
# img = transforms.Resize((224, 224))(img)
# img = transforms.ToTensor()(img)
# img = transforms.Normalize(mean=[...], std=[...])(img)

transform = transforms.Compose([
  transforms.Resize((224, 224)), # 입력 이미지의 크기를 강제 변환
  transforms.ToTensor(), # tensor화
  transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # 표준화
])

# # 훈련 dataset load
# train_dataset = datasets.ImageFolder(root='./train', transform=transform)
# train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# # test dataset load
# test_dataset = datasets.ImageFolder(root='./test', transform=transform)
# test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

# model = models.resnet50(pretrained=True)
# num_ftrs = model.fn.in_features
# model.fc = nn.Linear(num_ftrs, 2)

# criterion = nn.CrossEntropyLoss()
# optimizer = optim.Adam(model.parameters(), lr=0.00005)

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)

In [ ]:
import gradio as gr
def update(name):
  return f"Welcome to gradio, {name}"

# Block을 쓰면 interface보다 더 많은 custom web application demo를 만들 수 있다.
# block 객체를 with 문과 함께 사용
with gr.Blocks() as demo:
  gr.Markdown("아래의 글을 작성하고 **실행** 버튼을 눌러 결과를 확인합니다.")

  # gr.Row: 가로 layout 구성
  with gr.Row():
    inp = gr.Textbox(placeholder="이름이 무엇인가요?")
    out = gr.Textbox()

  btn = gr.Button("실행")
  btn.click(fn=update, inputs=inp, outputs=out)

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [6]:
from transformers import GPT2LMHeadModel
from transformers import PreTrainedTokenizerFast
from transformers import pipeline
# model download
# pipeline: pipeline module은 fine-tuning이 아닌 단순한 model 추론을 할 때 쉽게 활용 가능하고 다양한 자연어처리와 image 처리 작업을 수행할 수 있도록 도와줌
# text_classifier: pipeline module을 활용하여 문장 분류 model을 불러옴.
# tokenizer: 문장 분류 model을 위한 skt의 kogpt2 model에 맞는 한국어 tokenizer를 불러옴.
# text_generator: 문장 생성 model을 불러옴.
text_classifier = pipeline("text-classification", model='smilegate-ai/kor_unsmile')
tokenizer = PreTrainedTokenizerFast.from_pretrained("skt/kogpt2-base-v2", bos_token='</s>', eos_token='</s>', unk_token='<unk>', pad_token='<pad>', mask_token='<mask>')
text_generator = GPT2LMHeadModel.from_pretrained('skt/kogpt2-base-v2')

c:\Users\ABHCST9\Desktop\dev\workspace\next_ai\venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ABHCST9\.cache\huggingface\hub\models--smilegate-ai--kor_unsmile. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 45558.23it/s]
c:\Users\ABHCST9\Des

In [ ]:
import gradio as gr

def is_inappropriate(input):
  output = text_classifier(input)[0]['label']
  if output == 'clean':
    return True, ""
  else:
    return False, output

def generate_text(input):
  input_ids = tokenizer.encode(input, return_tensors='pt')
  gen_ids = text_generator.generate(input_ids,
                         max_length=128,
                         repetition_penalty=2.0,
                         pad_token_id=tokenizer.pad_token_id,
                         eos_token_id=tokenizer.eos_token_id,
                         bos_token_id=tokenizer.bos_token_id,
                         use_cache=True)
  generated = tokenizer.decode(gen_ids[0])
  return generated

def chatbot_response(input):
  if len(input) > 100:  # 문장 길이 제한
    return "오류: 문장이 너무 깁니다. 100자 이내로 제한해 주세요."

  is_appropriate, feedback = is_inappropriate(input)

  if not is_appropriate:
    return f"오류: 부적절한 내용이 포함되어 있습니다: {feedback}"

  generated = generate_text(input)
  return "응답: " + generated

def feedback_submission(feedback):
  # write feedback
  with open("feedback.txt", "a") as f:
    f.write(f"{feedback}" + "\n")
  return "피드백이 제출되었습니다. 감사합니다!"

def save_text(text):
  with open("generated.txt", "a") as f:
    f.write(text + "\n")
  return "텍스트가 저장되었습니다."

with gr.Blocks() as app:
  with gr.Row():
    with gr.Column():
      chat_input = gr.Textbox(label="앞 문장을 입력하세요.")
      chat_output = gr.Textbox(label="생성된 전체 문장", interactive=False)
      submit_button = gr.Button("제출")
      save_button = gr.Button("저장")
      save_feedback_label = gr.Label(label='생성 문장 저장 결과')
    with gr.Column():
      feedback_input = gr.Textbox(label="피드백")
      feedback_button = gr.Button("피드백 제출")
      feedback_feedback_label = gr.Label(label='피드백 제출 결과')  # 피드백 제출 성공 메시지 출력을 위한 레이블

  submit_button.click(chatbot_response, inputs=chat_input, outputs=chat_output)
  feedback_button.click(feedback_submission, inputs=feedback_input, outputs=feedback_feedback_label)
  save_button.click(save_text, inputs=chat_output, outputs=save_feedback_label)

app.launch(debug=True, share=True)


* Running on local URL:  http://127.0.0.1:7861

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


Keyboard interruption in main thread... closing server.


# huggingface 배포

In [ ]:
# 1. 로그인
# 2. profile에서 new space click
#    - space: 다른 사람이 내 model을 download하거나 내 app을 활용할 수 있게끔 도와주는 공간
# 3. 이름 설정, license 설정(mit, apache 등)
# 4. SDK(개발을 편하게 해주는 도구) 선택   
# 5. hardware 선택(free version)
# 6. click create button
# 7. 우측 상단에 files click
# 8. contribute click
# 9. create a new file을 선택하여 코드를 작성하거나, upload files를 선택해서 file들을 올리기
# 10. create a new file일 때는 file 이름 입력, upload files는 library 모음 file 같은 것 upload
# 11. click commit new file to main